<a href="https://colab.research.google.com/github/NicoMora827/Inteligencia-Artificial-2-/blob/main/Dataset_prediccion_de_valor_de_las_casas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "d39ad0b3",
   "metadata": {},
   "source": [
    "# Predicción del precio de casas\n",
    "\n",
    "**Objetivo:** construir un modelo predictivo para estimar el precio de una casa y comparar dos modelos: **Regresión Lineal** y **Árbol de Decisión**.\n",
    "\n",
    "El dataset utilizado corresponde al conjunto de datos de precios de viviendas de Kaggle, que contiene 545 registros y variables como área, habitaciones, baños, pisos, parqueaderos y características adicionales de la vivienda. citeturn0search4\n",
    "\n",
    "Los comentarios del código están escritos en primera persona y explican qué entiendo que hace cada línea, para que el notebook sirva también como evidencia del razonamiento personal."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2042e164",
   "metadata": {},
   "source": [
    "## 1. Importación de librerías\n",
    "\n",
    "Voy a utilizar pandas y numpy para trabajar con los datos, matplotlib y seaborn para las gráficas y scikit-learn para preparar los datos, entrenar los modelos y medir su rendimiento."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "2ebca27d",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Importo pandas porque lo voy a utilizar para leer y organizar la información del archivo CSV.\n",
    "import pandas as pd\n",
    "# Importo numpy porque me sirve para realizar operaciones numéricas cuando sean necesarias.\n",
    "import numpy as np\n",
    "# Importo matplotlib porque quiero representar visualmente algunos resultados.\n",
    "import matplotlib.pyplot as plt\n",
    "# Importo seaborn porque me ayuda a crear gráficas estadísticas de una forma sencilla.\n",
    "import seaborn as sns\n",
    "# Importo train_test_split porque necesito separar los datos para entrenar y evaluar los modelos.\n",
    "from sklearn.model_selection import train_test_split\n",
    "# Importo ColumnTransformer porque necesito aplicar transformaciones diferentes a columnas numéricas y categóricas.\n",
    "from sklearn.compose import ColumnTransformer\n",
    "# Importo OneHotEncoder porque las variables que tienen texto deben convertirse a valores numéricos.\n",
    "from sklearn.preprocessing import OneHotEncoder\n",
    "# Importo Pipeline porque quiero unir la preparación de datos y el modelo en un mismo proceso.\n",
    "from sklearn.pipeline import Pipeline\n",
    "# Importo LinearRegression porque este será uno de los modelos que voy a comparar.\n",
    "from sklearn.linear_model import LinearRegression\n",
    "# Importo DecisionTreeRegressor porque este será el segundo modelo de predicción.\n",
    "from sklearn.tree import DecisionTreeRegressor\n",
    "# Importo las métricas que voy a utilizar para comparar los resultados de ambos modelos.\n",
    "from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d8192293",
   "metadata": {},
   "source": [
    "## 2. Cargar el dataset\n",
    "\n",
    "En Colab se puede subir directamente el archivo `Housing.csv`. Esto evita depender de credenciales de Kaggle dentro del notebook."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6ccd6477",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Importo la herramienta de Colab que me permite seleccionar el archivo desde mi computador.\n",
    "from google.colab import files\n",
    "# Abro el selector de archivos para cargar el dataset.\n",
    "uploaded = files.upload()\n",
    "# Tomo el nombre del primer archivo que seleccioné para poder leerlo.\n",
    "archivo = next(iter(uploaded))\n",
    "# Leo el archivo CSV y lo convierto en un DataFrame para trabajar con él.\n",
    "df = pd.read_csv(archivo)\n",
    "# Muestro las primeras filas para comprobar que el dataset se cargó correctamente.\n",
    "df.head()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "7f1493b3",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Muestro la cantidad de filas y columnas porque quiero saber el tamaño del conjunto de datos.\n",
    "print(\"Tamaño del dataset:\", df.shape)\n",
    "# Muestro los nombres de las columnas para identificar las variables que tengo disponibles.\n",
    "print(\"Columnas:\", df.columns.tolist())\n",
    "# Reviso los tipos de datos para saber cuáles variables son numéricas y cuáles son de texto.\n",
    "df.info()\n",
    "# Compruebo cuántos valores faltantes existen en cada columna antes de entrenar los modelos.\n",
    "print(\"\\nValores faltantes por columna:\")\n",
    "print(df.isnull().sum())\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fbcb7386",
   "metadata": {},
   "source": [
    "## 3. Exploración inicial\n",
    "\n",
    "La variable que quiero predecir es `price`. El resto de variables serán utilizadas como características de la vivienda.\n",
    "\n",
    "El dataset incluye variables numéricas y categóricas, por eso no voy a convertir todo manualmente con números arbitrarios. Para las variables categóricas usaré codificación One-Hot, que permite representar categorías como columnas binarias."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "52b642c2",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Muestro un resumen estadístico para conocer rangos, promedios y posibles valores extremos.\n",
    "df.describe()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f0f43a5f",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Creo una gráfica de la distribución de precios para observar cómo están repartidos los valores.\n",
    "plt.figure(figsize=(8, 5))\n",
    "# Uso un histograma porque quiero ver en qué rangos se concentran los precios.\n",
    "sns.histplot(df[\"price\"], kde=True)\n",
    "# Coloco un título para dejar claro qué representa la gráfica.\n",
    "plt.title(\"Distribución de precios de las viviendas\")\n",
    "# Indico que el eje horizontal representa el precio.\n",
    "plt.xlabel(\"Precio\")\n",
    "# Indico que el eje vertical representa la cantidad de viviendas.\n",
    "plt.ylabel(\"Cantidad de viviendas\")\n",
    "# Muestro la gráfica.\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e3f304d9",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Creo una matriz de correlación únicamente con las variables numéricas para observar relaciones con el precio.\n",
    "correlacion = df.select_dtypes(include=np.number).corr()\n",
    "# Creo una figura suficientemente grande para leer los nombres de las variables.\n",
    "plt.figure(figsize=(9, 7))\n",
    "# Muestro la matriz como un mapa de calor para facilitar la interpretación.\n",
    "sns.heatmap(correlacion, annot=True, cmap=\"coolwarm\", fmt=\".2f\")\n",
    "# Coloco un título para explicar qué estoy observando.\n",
    "plt.title(\"Correlación entre variables numéricas\")\n",
    "# Muestro la gráfica.\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "75fdc428",
   "metadata": {},
   "source": [
    "## 4. Separación de variables\n",
    "\n",
    "Voy a utilizar `price` como variable objetivo porque es el valor que quiero predecir.\n",
    "\n",
    "Las demás columnas serán las variables de entrada. Después separaré los datos en 80 % para entrenamiento y 20 % para prueba. Esta separación permite evaluar el modelo con datos que no utilizó durante el entrenamiento."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "df6cf737",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Guardo en X todas las columnas excepto price porque son las características que utilizaré para predecir.\n",
    "X = df.drop(\"price\", axis=1)\n",
    "# Guardo en y solamente price porque representa el resultado que quiero que el modelo aprenda a predecir.\n",
    "y = df[\"price\"]\n",
    "# Separo los datos dejando el 80 % para aprender y el 20 % para comprobar qué tan bien generaliza el modelo.\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)\n",
    "# Muestro los tamaños para comprobar que la división se realizó como esperaba.\n",
    "print(\"Datos de entrenamiento:\", X_train.shape)\n",
    "print(\"Datos de prueba:\", X_test.shape)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0aeac090",
   "metadata": {},
   "source": [
    "## 5. Preparación de variables\n",
    "\n",
    "Primero identifico las columnas numéricas y las que contienen texto.\n",
    "\n",
    "Las columnas numéricas pueden pasar directamente a los modelos. Las columnas categóricas necesitan convertirse a números mediante One-Hot Encoding."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "66d7eb09",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Busco automáticamente las columnas numéricas para no tener que escribirlas una por una.\n",
    "columnas_numericas = X.select_dtypes(include=[\"int64\", \"float64\"]).columns.tolist()\n",
    "# Busco las columnas de texto porque son las que necesitan una transformación.\n",
    "columnas_categoricas = X.select_dtypes(include=[\"object\"]).columns.tolist()\n",
    "# Muestro las columnas numéricas para comprobar que fueron identificadas correctamente.\n",
    "print(\"Columnas numéricas:\", columnas_numericas)\n",
    "# Muestro las columnas categóricas para comprobar que fueron identificadas correctamente.\n",
    "print(\"Columnas categóricas:\", columnas_categoricas)\n",
    "# Creo el transformador que convierte las categorías en columnas binarias y evita problemas con categorías desconocidas.\n",
    "preprocesador = ColumnTransformer(\n",
    "    transformers=[\n",
    "        (\"categoricas\", OneHotEncoder(handle_unknown=\"ignore\", drop=\"first\"), columnas_categoricas)\n",
    "    ],\n",
    "    remainder=\"passthrough\"\n",
    ")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a5b8ad2b",
   "metadata": {},
   "source": [
    "## 6. Modelo de Regresión Lineal\n",
    "\n",
    "La regresión lineal será mi primer modelo de referencia. La idea es encontrar una relación matemática entre las características de la vivienda y su precio.\n",
    "\n",
    "No espero que una casa siga una relación perfectamente lineal, pero este modelo me sirve como punto de comparación frente al árbol de decisión."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b23ad063",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Creo un pipeline para que primero se transformen las categorías y después se entrene la regresión lineal.\n",
    "modelo_lineal = Pipeline(steps=[\n",
    "    (\"preprocesamiento\", preprocesador),\n",
    "    (\"modelo\", LinearRegression())\n",
    "])\n",
    "# Entreno la regresión lineal usando solamente los datos destinados al entrenamiento.\n",
    "modelo_lineal.fit(X_train, y_train)\n",
    "# Genero predicciones sobre los datos de prueba que el modelo no vio durante el entrenamiento.\n",
    "pred_lineal = modelo_lineal.predict(X_test)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "224483b8",
   "metadata": {},
   "source": [
    "## 7. Modelo de Árbol de Decisión\n",
    "\n",
    "El árbol de decisión puede dividir los datos en diferentes condiciones. Esto me permite representar relaciones no lineales que una regresión lineal podría no capturar tan bien.\n",
    "\n",
    "Voy a limitar la profundidad del árbol para evitar que memorice demasiado los datos de entrenamiento."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6b3068fe",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Creo el árbol con una profundidad máxima para controlar su complejidad.\n",
    "modelo_arbol = Pipeline(steps=[\n",
    "    (\"preprocesamiento\", preprocesador),\n",
    "    (\"modelo\", DecisionTreeRegressor(max_depth=5, random_state=42))\n",
    "])\n",
    "# Entreno el árbol utilizando los datos de entrenamiento.\n",
    "modelo_arbol.fit(X_train, y_train)\n",
    "# Genero las predicciones del árbol sobre los datos de prueba.\n",
    "pred_arbol = modelo_arbol.predict(X_test)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "9e21bcc8",
   "metadata": {},
   "source": [
    "## 8. Evaluación de los modelos\n",
    "\n",
    "Voy a comparar los modelos usando tres medidas:\n",
    "\n",
    "- **MAE:** error absoluto promedio; entre menor sea, mejor.\n",
    "- **RMSE:** penaliza más los errores grandes; entre menor sea, mejor.\n",
    "- **R²:** indica qué proporción de la variación del precio logra explicar el modelo; mientras más cercano a 1, mejor.\n",
    "\n",
    "Estas métricas son apropiadas para un problema de regresión porque el resultado que estoy prediciendo es un valor numérico continuo. citeturn0search12"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e191b684",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Creo una función para calcular las mismas métricas para cada modelo y no repetir código innecesariamente.\n",
    "def evaluar_modelo(nombre, real, predicho):\n",
    "    # Calculo el error absoluto promedio para saber cuánto se alejan las predicciones del precio real en promedio.\n",
    "    mae = mean_absolute_error(real, predicho)\n",
    "    # Calculo el error cuadrático medio para dar más importancia a errores grandes.\n",
    "    mse = mean_squared_error(real, predicho)\n",
    "    # Calculo la raíz del error cuadrático medio para que el resultado quede en la misma unidad del precio.\n",
    "    rmse = np.sqrt(mse)\n",
    "    # Calculo R2 para saber qué tan bien explica el modelo las variaciones de los precios.\n",
    "    r2 = r2_score(real, predicho)\n",
    "    # Devuelvo los resultados organizados para poder compararlos después.\n",
    "    return [nombre, mae, rmse, r2]\n",
    "# Evalúo la regresión lineal usando los precios reales y sus predicciones.\n",
    "resultado_lineal = evaluar_modelo(\"Regresión Lineal\", y_test, pred_lineal)\n",
    "# Evalúo el árbol de decisión usando los mismos datos de prueba.\n",
    "resultado_arbol = evaluar_modelo(\"Árbol de Decisión\", y_test, pred_arbol)\n",
    "# Creo una tabla con los resultados de los dos modelos para compararlos fácilmente.\n",
    "resultados = pd.DataFrame(\n",
    "    [resultado_lineal, resultado_arbol],\n",
    "    columns=[\"Modelo\", \"MAE\", \"RMSE\", \"R2\"]\n",
    ")\n",
    "# Ordeno los modelos por R2 de mayor a menor para identificar cuál tuvo mejor capacidad explicativa.\n",
    "resultados.sort_values(\"R2\", ascending=False)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "064b22bf",
   "metadata": {},
   "source": [
    "## 9. Comparación visual\n",
    "\n",
    "Ahora quiero comprobar visualmente qué tan cerca están las predicciones de los precios reales. Si los puntos se acercan a una línea diagonal imaginaria, significa que las predicciones están más cerca de los valores reales."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "15d3b83b",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Creo una figura para comparar los valores reales con las predicciones del modelo lineal.\n",
    "plt.figure(figsize=(7, 5))\n",
    "# Grafico cada precio real frente al precio que predijo la regresión lineal.\n",
    "plt.scatter(y_test, pred_lineal, alpha=0.6)\n",
    "# Agrego una línea de referencia para representar el caso de una predicción perfecta.\n",
    "plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle=\"--\")\n",
    "# Coloco un título para identificar el modelo analizado.\n",
    "plt.title(\"Regresión Lineal: valores reales vs predicciones\")\n",
    "# Indico que el eje horizontal contiene los valores reales.\n",
    "plt.xlabel(\"Precio real\")\n",
    "# Indico que el eje vertical contiene los valores predichos.\n",
    "plt.ylabel(\"Precio predicho\")\n",
    "# Muestro la gráfica.\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "3aa43e08",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Creo una figura para comparar los valores reales con las predicciones del árbol de decisión.\n",
    "plt.figure(figsize=(7, 5))\n",
    "# Grafico cada precio real frente al precio que predijo el árbol.\n",
    "plt.scatter(y_test, pred_arbol, alpha=0.6)\n",
    "# Agrego la misma línea de referencia para poder interpretar visualmente el resultado.\n",
    "plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle=\"--\")\n",
    "# Coloco un título para identificar el segundo modelo.\n",
    "plt.title(\"Árbol de Decisión: valores reales vs predicciones\")\n",
    "# Indico que el eje horizontal contiene los valores reales.\n",
    "plt.xlabel(\"Precio real\")\n",
    "# Indico que el eje vertical contiene los valores predichos.\n",
    "plt.ylabel(\"Precio predicho\")\n",
    "# Muestro la gráfica.\n",
    "plt.show()\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4ea52147",
   "metadata": {},
   "source": [
    "## 10. Análisis de resultados\n",
    "\n",
    "Después de ejecutar el notebook, debo mirar principalmente el R², MAE y RMSE.\n",
    "\n",
    "Mi interpretación será:\n",
    "\n",
    "- Si un modelo tiene **R² más alto**, logra explicar mejor la variación de los precios.\n",
    "- Si tiene **MAE y RMSE más bajos**, sus errores de predicción son menores.\n",
    "- Si el árbol supera a la regresión lineal, puedo concluir que en este conjunto de datos las relaciones entre las características y el precio tienen componentes no lineales que el árbol consigue representar mejor.\n",
    "- Si la regresión lineal supera al árbol, significa que una relación más simple fue suficiente para este conjunto de datos.\n",
    "\n",
    "No debo afirmar cuál modelo ganó antes de ejecutar el notebook, porque los resultados dependen de la partición de los datos y de los parámetros utilizados."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c4482533",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Busco automáticamente el modelo con el R2 más alto porque esa será mi referencia principal para elegir el ganador.\n",
    "mejor_modelo = resultados.loc[resultados[\"R2\"].idxmax(), \"Modelo\"]\n",
    "# Muestro el nombre del modelo que obtuvo el mayor R2.\n",
    "print(\"Modelo con mayor R2:\", mejor_modelo)\n",
    "# Busco el valor de R2 del modelo ganador para poder interpretarlo.\n",
    "mejor_r2 = resultados.loc[resultados[\"R2\"].idxmax(), \"R2\"]\n",
    "# Muestro el R2 con cuatro decimales para que sea fácil de leer.\n",
    "print(\"R2 del mejor modelo:\", round(mejor_r2, 4))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f168fffe",
   "metadata": {},
   "source": [
    "## 11. Predicción de una casa nueva\n",
    "\n",
    "Finalmente voy a dejar un ejemplo donde puedo introducir las características de una vivienda y obtener una estimación de precio.\n",
    "\n",
    "Este ejemplo sirve para demostrar que el proyecto no solamente compara modelos, sino que también puede utilizar el modelo entrenado para realizar una predicción."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "6b0465be",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Creo los datos de una vivienda de ejemplo manteniendo exactamente los nombres de columnas del dataset.\n",
    "casa_nueva = pd.DataFrame([{\n",
    "    \"area\": 6000,\n",
    "    \"bedrooms\": 3,\n",
    "    \"bathrooms\": 2,\n",
    "    \"stories\": 2,\n",
    "    \"mainroad\": \"yes\",\n",
    "    \"guestroom\": \"no\",\n",
    "    \"basement\": \"no\",\n",
    "    \"hotwaterheating\": \"no\",\n",
    "    \"airconditioning\": \"yes\",\n",
    "    \"parking\": 2,\n",
    "    \"prefarea\": \"yes\",\n",
    "    \"furnishingstatus\": \"semi-furnished\"\n",
    "}])\n",
    "# Uso la regresión lineal para estimar el precio de la vivienda de ejemplo.\n",
    "prediccion_lineal_nueva = modelo_lineal.predict(casa_nueva)[0]\n",
    "# Uso el árbol de decisión para obtener una segunda estimación de la misma vivienda.\n",
    "prediccion_arbol_nueva = modelo_arbol.predict(casa_nueva)[0]\n",
    "# Muestro la predicción de la regresión lineal con formato de dos decimales.\n",
    "print(\"Predicción con Regresión Lineal:\", round(prediccion_lineal_nueva, 2))\n",
    "# Muestro la predicción del árbol de decisión con formato de dos decimales.\n",
    "print(\"Predicción con Árbol de Decisión:\", round(prediccion_arbol_nueva, 2))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "53861d2b",
   "metadata": {},
   "source": [
    "## Conclusión\n",
    "\n",
    "En este proyecto construí dos modelos para predecir el precio de las viviendas: regresión lineal y árbol de decisión. Primero preparé las variables categóricas, luego dividí los datos en entrenamiento y prueba y finalmente comparé ambos modelos mediante MAE, RMSE y R².\n",
    "\n",
    "La conclusión definitiva debe basarse en los resultados obtenidos al ejecutar el notebook. De esta forma no estoy escogiendo un modelo por intuición, sino por su rendimiento medido sobre datos de prueba.\n",
    "\n",
    "**Fuente del dataset:** Kaggle, Housing Prices Dataset. citeturn0search4"
   ]
  }
 ],
 "metadata": {
  "colab": {
   "name": "Modelo_Predictivo_Precios_Casas.ipynb",
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
